In [1]:
from datetime import datetime
import sqlite3
import tkinter as tk
from tkinter import messagebox, ttk

DB_NAME = "prosthetics_clinic.db"


# ==========================================
# 1. DATABASE MANAGEMENT & INITIALIZATION
# ==========================================
def get_db_connection():
    conn = sqlite3.connect(DB_NAME)
    # Mandated: SQLite requires explicit foreign key enabling per connection
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn


def init_db():
    conn = get_db_connection()
    cursor = conn.cursor()

    # Create Tables with Cascading Deletes
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS clinics (
            clinic_id INTEGER PRIMARY KEY AUTOINCREMENT,
            clinic_name TEXT UNIQUE NOT NULL,
            contact_email TEXT NOT NULL
        )
    """
    )

    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS patient_fittings (
            fitting_id INTEGER PRIMARY KEY AUTOINCREMENT,
            clinic_id INTEGER NOT NULL,
            patient_name TEXT NOT NULL,
            fitting_date TEXT NOT NULL,
            FOREIGN KEY (clinic_id) REFERENCES clinics(clinic_id) ON DELETE CASCADE
        )
    """
    )

    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS component_costs (
            cost_id INTEGER PRIMARY KEY AUTOINCREMENT,
            fitting_id INTEGER NOT NULL,
            component_description TEXT NOT NULL,
            cost_amount REAL NOT NULL,
            FOREIGN KEY (fitting_id) REFERENCES patient_fittings(fitting_id) ON DELETE CASCADE
        )
    """
    )

    # Automatically seed database if empty
    cursor.execute("SELECT COUNT(*) FROM clinics")
    if cursor.fetchone()[0] == 0:
        cursor.executemany(
            """
            INSERT INTO clinics (clinic_name, contact_email) VALUES (?, ?)
        """,
            [
                ("Apex Prosthetics - North", "north@apexprosthetics.com"),
                ("Apex Prosthetics - South", "south@apexprosthetics.com"),
            ],
        )

        cursor.executemany(
            """
            INSERT INTO patient_fittings (clinic_id, patient_name, fitting_date) VALUES (?, ?, ?)
        """,
            [
                (1, "Deepa Iyer", "2026-03-15"),
                (2, "Mohit Chauhan", "2026-03-18"),
            ],
        )

        cursor.executemany(
            """
            INSERT INTO component_costs (fitting_id, component_description, cost_amount) VALUES (?, ?, ?)
        """,
            [
                (1, "Socket", 1200.50),
                (1, "Joint Mechanism", 850.00),
                (2, "Cosmetic Covering", 340.75),
            ],
        )

    conn.commit()
    conn.close()


# ==========================================
# 2. GUI APPLICATION
# ==========================================
class ProstheticsApp:

    def __init__(self, root):
        self.root = root
        self.root.title("Prosthetics Clinic & Patient Fitting Dashboard")
        self.root.geometry("950x650")

        # Styling
        style = ttk.Style()
        style.theme_use("clam")

        # Main Layout Containers
        top_frame = ttk.Frame(self.root, padding=10)
        top_frame.pack(side=tk.TOP, fill=tk.X)

        bottom_frame = ttk.Frame(self.root, padding=10)
        bottom_frame.pack(side=tk.BOTTOM, fill=tk.BOTH, expand=True)

        # Build GUI Panels
        self.build_insert_panel(top_frame)
        self.build_delete_panel(top_frame)
        self.build_tabbed_display(bottom_frame)

        # Initial Load Data into Tables & Dropdowns
        self.refresh_all()

    # --------------------------------------
    # INSERT PANEL
    # --------------------------------------
    def build_insert_panel(self, parent):
        panel = ttk.LabelFrame(
            parent, text=" Insert New Fitting Record ", padding=10
        )
        panel.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=5)

        ttk.Label(panel, text="Clinic Name:").grid(
            row=0, column=0, sticky=tk.W, pady=2
        )
        self.ent_clinic = ttk.Entry(panel, width=22)
        self.ent_clinic.grid(row=0, column=1, pady=2, padx=5)

        ttk.Label(panel, text="Patient Name:").grid(
            row=1, column=0, sticky=tk.W, pady=2
        )
        self.ent_patient = ttk.Entry(panel, width=22)
        self.ent_patient.grid(row=1, column=1, pady=2, padx=5)

        ttk.Label(panel, text="Component Description:").grid(
            row=2, column=0, sticky=tk.W, pady=2
        )
        self.ent_component = ttk.Entry(panel, width=22)
        self.ent_component.grid(row=2, column=1, pady=2, padx=5)

        ttk.Label(panel, text="Cost Amount ($):").grid(
            row=3, column=0, sticky=tk.W, pady=2
        )
        self.ent_cost = ttk.Entry(panel, width=22)
        self.ent_cost.grid(row=3, column=1, pady=2, padx=5)

        btn_add = ttk.Button(
            panel, text="Add Record", command=self.add_record
        )
        btn_add.grid(row=4, column=0, columnspan=2, pady=8)

    def add_record(self):
        clinic = self.ent_clinic.get().strip()
        patient = self.ent_patient.get().strip()
        component = self.ent_component.get().strip()
        cost_raw = self.ent_cost.get().strip()

        # Validation: Non-blank check
        if not clinic or not patient or not component or not cost_raw:
            messagebox.showerror(
                "Input Error", "All fields are required. Please fill them out."
            )
            return

        # Validation: Positive Numeric Check
        try:
            cost = float(cost_raw)
            if cost <= 0:
                raise ValueError
        except ValueError:
            messagebox.showerror(
                "Input Error",
                "Cost Amount must be a positive number greater than 0.",
            )
            return

        conn = get_db_connection()
        cursor = conn.cursor()

        try:
            # 1. Check or Create Clinic
            cursor.execute(
                "SELECT clinic_id FROM clinics WHERE clinic_name = ?", (clinic,)
            )
            row = cursor.fetchone()
            if row:
                clinic_id = row[0]
            else:
                default_email = (
                    f"contact@{clinic.lower().replace(' ', '')}.com"
                )
                cursor.execute(
                    "INSERT INTO clinics (clinic_name, contact_email) VALUES (?, ?)",
                    (clinic, default_email),
                )
                clinic_id = cursor.lastrowid

            # 2. Insert Patient Fitting
            today_str = datetime.today().strftime("%Y-%m-%d")
            cursor.execute(
                "INSERT INTO patient_fittings (clinic_id, patient_name, fitting_date) VALUES (?, ?, ?)",
                (clinic_id, patient, today_str),
            )
            fitting_id = cursor.lastrowid

            # 3. Insert Component Cost
            cursor.execute(
                "INSERT INTO component_costs (fitting_id, component_description, cost_amount) VALUES (?, ?, ?)",
                (fitting_id, component, cost),
            )

            conn.commit()
            messagebox.showinfo(
                "Success", "Record added successfully to database."
            )

            # Clear inputs
            self.ent_clinic.delete(0, tk.END)
            self.ent_patient.delete(0, tk.END)
            self.ent_component.delete(0, tk.END)
            self.ent_cost.delete(0, tk.END)

            self.refresh_all()

        except sqlite3.Error as e:
            conn.rollback()
            messagebox.showerror("Database Error", f"An error occurred: {e}")
        finally:
            conn.close()

    # --------------------------------------
    # DELETE PANEL
    # --------------------------------------
    def build_delete_panel(self, parent):
        panel = ttk.LabelFrame(
            parent, text=" Delete Record Panel ", padding=10
        )
        panel.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True, padx=5)

        ttk.Label(panel, text="Target Entity:").grid(
            row=0, column=0, sticky=tk.W, pady=5
        )
        self.cb_entity = ttk.Combobox(
            panel,
            values=["Clinic", "Patient Fitting", "Component Cost"],
            state="readonly",
            width=22,
        )
        self.cb_entity.current(0)
        self.cb_entity.grid(row=0, column=1, pady=5, padx=5)
        self.cb_entity.bind("<<ComboboxSelected>>", self.update_delete_dropdown)

        ttk.Label(panel, text="Select Item:").grid(
            row=1, column=0, sticky=tk.W, pady=5
        )
        self.cb_item = ttk.Combobox(panel, state="readonly", width=35)
        self.cb_item.grid(row=1, column=1, pady=5, padx=5)

        btn_delete = ttk.Button(
            panel, text="Delete Selected", command=self.delete_record
        )
        btn_delete.grid(row=2, column=0, columnspan=2, pady=15)

    def update_delete_dropdown(self, event=None):
        entity = self.cb_entity.get()
        conn = get_db_connection()
        cursor = conn.cursor()

        options = []
        self.item_map = {}

        if entity == "Clinic":
            cursor.execute(
                "SELECT clinic_id, clinic_name FROM clinics ORDER BY clinic_name"
            )
            for cid, name in cursor.fetchall():
                label = f"ID {cid} - {name}"
                options.append(label)
                self.item_map[label] = cid

        elif entity == "Patient Fitting":
            cursor.execute(
                "SELECT fitting_id, patient_name, fitting_date FROM patient_fittings ORDER BY fitting_id"
            )
            for fid, name, date in cursor.fetchall():
                label = f"Fitting #{fid} - Patient: {name} ({date})"
                options.append(label)
                self.item_map[label] = fid

        elif entity == "Component Cost":
            cursor.execute(
                "SELECT cost_id, component_description, cost_amount FROM component_costs ORDER BY cost_id"
            )
            for cost_id, desc, cost in cursor.fetchall():
                label = f"Cost #{cost_id} - {desc} (${cost:.2f})"
                options.append(label)
                self.item_map[label] = cost_id

        conn.close()

        self.cb_item["values"] = options
        if options:
            self.cb_item.current(0)
        else:
            self.cb_item.set("")

    def delete_record(self):
        selected_label = self.cb_item.get()
        if not selected_label or selected_label not in self.item_map:
            messagebox.showwarning(
                "Selection Error", "Please select a valid item to delete."
            )
            return

        item_id = self.item_map[selected_label]
        entity = self.cb_entity.get()

        confirm = messagebox.askyesno(
            "Confirm Deletion",
            f"Are you sure you want to delete this {entity}?\n(Associated child records will be cascade deleted automatically).",
        )
        if not confirm:
            return

        conn = get_db_connection()
        cursor = conn.cursor()

        try:
            if entity == "Clinic":
                cursor.execute(
                    "DELETE FROM clinics WHERE clinic_id = ?", (item_id,)
                )
            elif entity == "Patient Fitting":
                cursor.execute(
                    "DELETE FROM patient_fittings WHERE fitting_id = ?",
                    (item_id,),
                )
            elif entity == "Component Cost":
                cursor.execute(
                    "DELETE FROM component_costs WHERE cost_id = ?", (item_id,)
                )

            conn.commit()
            messagebox.showinfo("Success", f"{entity} deleted successfully.")
            self.refresh_all()

        except sqlite3.Error as e:
            conn.rollback()
            messagebox.showerror("Database Error", f"Deletion failed: {e}")
        finally:
            conn.close()

    # --------------------------------------
    # TABBED DISPLAY PANEL
    # --------------------------------------
    def build_tabbed_display(self, parent):
        notebook = ttk.Notebook(parent)
        notebook.pack(fill=tk.BOTH, expand=True)

        # Tab 1: JOIN View
        self.tree_join = self.create_tree_tab(
            notebook,
            "Combined View",
            [
                "Fitting ID",
                "Clinic Name",
                "Patient Name",
                "Fitting Date",
                "Component",
                "Cost ($)",
            ],
        )

        # Tab 2: Clinics Table
        self.tree_clinics = self.create_tree_tab(
            notebook,
            "Clinics Table",
            ["Clinic ID", "Clinic Name", "Contact Email"],
        )

        # Tab 3: Patient Fittings Table
        self.tree_fittings = self.create_tree_tab(
            notebook,
            "Patient Fittings Table",
            ["Fitting ID", "Clinic ID", "Patient Name", "Fitting Date"],
        )

        # Tab 4: Component Costs Table
        self.tree_costs = self.create_tree_tab(
            notebook,
            "Component Costs Table",
            ["Cost ID", "Fitting ID", "Component Description", "Cost ($)"],
        )

    def create_tree_tab(self, notebook, tab_name, columns):
        frame = ttk.Frame(notebook)
        notebook.add(frame, text=tab_name)

        tree = ttk.Treeview(frame, columns=columns, show="headings")
        for col in columns:
            tree.heading(col, text=col)
            tree.column(col, anchor=tk.CENTER, width=120)

        # Scrollbar integration
        scrollbar = ttk.Scrollbar(
            frame, orient=tk.VERTICAL, command=tree.yview
        )
        tree.configure(yscrollcommand=scrollbar.set)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        tree.pack(fill=tk.BOTH, expand=True)

        return tree

    # --------------------------------------
    # REFRESH / STATE MANAGEMENT
    # --------------------------------------
    def refresh_all(self):
        self.refresh_tables()
        self.update_delete_dropdown()

    def populate_tree(self, tree, rows):
        for row in tree.get_children():
            tree.delete(row)
        for row in rows:
            tree.insert("", tk.END, values=row)

    def refresh_tables(self):
        conn = get_db_connection()
        cursor = conn.cursor()

        # Tab 1 JOIN query
        join_query = """
            SELECT pf.fitting_id, c.clinic_name, pf.patient_name, pf.fitting_date, cc.component_description, cc.cost_amount
            FROM patient_fittings pf
            JOIN clinics c ON pf.clinic_id = c.clinic_id
            JOIN component_costs cc ON pf.fitting_id = cc.fitting_id
            ORDER BY pf.fitting_id DESC
        """
        cursor.execute(join_query)
        self.populate_tree(self.tree_join, cursor.fetchall())

        # Tab 2 Clinics
        cursor.execute("SELECT * FROM clinics ORDER BY clinic_id")
        self.populate_tree(self.tree_clinics, cursor.fetchall())

        # Tab 3 Fittings
        cursor.execute("SELECT * FROM patient_fittings ORDER BY fitting_id")
        self.populate_tree(self.tree_fittings, cursor.fetchall())

        # Tab 4 Costs
        cursor.execute("SELECT * FROM component_costs ORDER BY cost_id")
        self.populate_tree(self.tree_costs, cursor.fetchall())

        conn.close()


# ==========================================
# 3. ENTRY POINT
# ==========================================
if __name__ == "__main__":
    init_db()
    root = tk.Tk()
    app = ProstheticsApp(root)
    root.mainloop()
 
 